<a href="https://colab.research.google.com/github/shivamkumar359/03-rd-SEM--ASSIGNMENTS/blob/main/IDB_W10_Q1_ASSIGNMENT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# TASK 1: IMPLEMENT LOG-BASED UNDO AND REDO OPERATIONS
# Google Colab - Complete Assignment Solution
# ============================================================

import os
import sys
import subprocess
from pathlib import Path



In [ ]:
# Create assignment directory
BASE_DIR = Path("/content")
os.chdir(BASE_DIR)

# ============================================================
# FILE 1: log_input.txt
# ============================================================

log_content = """<T1 start>
<T1, X, 100, 150>
<T1 commit>

<T2 start>
<T2, Y, 200, 250>

<T3 start>
<T3, Z, 300, 350>
<T3 commit>
"""

with open("log_input.txt", "w") as file:
    file.write(log_content)


# ============================================================
# FILE 2: recovery_manager.py
# ============================================================

python_code = r'''
import re
from pathlib import Path


def read_logs(filename):
    """Read and parse transaction logs."""

    transactions = {}
    updates = []

    with open(filename, "r") as file:
        for line in file:

            line = line.strip()

            if not line:
                continue

            # Remove angle brackets
            line = line.strip("<>")

            # Transaction START record
            if "start" in line.lower():
                transaction_id = line.split()[0]

                transactions.setdefault(
                    transaction_id,
                    {"status": "active"}
                )

            # Transaction COMMIT record
            elif "commit" in line.lower():
                transaction_id = line.split()[0]

                transactions.setdefault(
                    transaction_id,
                    {"status": "active"}
                )

                transactions[transaction_id]["status"] = "committed"

            # Transaction ABORT record
            elif "abort" in line.lower():
                transaction_id = line.split()[0]

                transactions.setdefault(
                    transaction_id,
                    {"status": "active"}
                )

                transactions[transaction_id]["status"] = "aborted"

            # Transaction UPDATE record
            else:
                parts = [p.strip() for p in line.split(",")]

                if len(parts) != 4:
                    raise ValueError(
                        f"Invalid log record: {line}"
                    )

                transaction_id = parts[0]
                variable = parts[1]
                old_value = int(parts[2])
                new_value = int(parts[3])

                transactions.setdefault(
                    transaction_id,
                    {"status": "active"}
                )

                updates.append({
                    "transaction": transaction_id,
                    "variable": variable,
                    "old": old_value,
                    "new": new_value
                })

    return transactions, updates


def recover_database(filename):

    print("=" * 55)
    print("       DATABASE RECOVERY MANAGER")
    print("=" * 55)

    transactions, updates = read_logs(filename)

    # --------------------------------------------------------
    # Step 1: Initialize database using original old values
    # --------------------------------------------------------

    database = {}

    for update in updates:
        variable = update["variable"]

        if variable not in database:
            database[variable] = update["old"]

    print("\nINITIAL DATABASE STATE:")
    print(database)

    # --------------------------------------------------------
    # Step 2: Identify transactions
    # --------------------------------------------------------

    committed = []
    incomplete = []
    aborted = []

    for transaction_id, info in transactions.items():

        status = info["status"]

        if status == "committed":
            committed.append(transaction_id)

        elif status == "active":
            incomplete.append(transaction_id)

        elif status == "aborted":
            aborted.append(transaction_id)

    print("\nTRANSACTION ANALYSIS:")
    print("Committed transactions:", committed)
    print("Incomplete transactions:", incomplete)
    print("Aborted transactions:", aborted)

    # --------------------------------------------------------
    # Step 3: REDO committed transactions
    # --------------------------------------------------------

    print("\n" + "=" * 55)
    print("REDO PHASE")
    print("=" * 55)

    for update in updates:

        transaction_id = update["transaction"]

        if transaction_id in committed:

            variable = update["variable"]
            new_value = update["new"]

            database[variable] = new_value

            print(
                f"REDO: {transaction_id} | "
                f"{variable}: {new_value}"
            )

    # --------------------------------------------------------
    # Step 4: UNDO incomplete transactions
    # Process updates in reverse order.
    # --------------------------------------------------------

    print("\n" + "=" * 55)
    print("UNDO PHASE")
    print("=" * 55)

    for update in reversed(updates):

        transaction_id = update["transaction"]

        if transaction_id in incomplete:

            variable = update["variable"]
            old_value = update["old"]

            database[variable] = old_value

            print(
                f"UNDO: {transaction_id} | "
                f"{variable}: {old_value}"
            )

    # --------------------------------------------------------
    # Step 5: Display final recovered database
    # --------------------------------------------------------

    print("\n" + "=" * 55)
    print("FINAL RECOVERED DATABASE STATE")
    print("=" * 55)

    for variable in sorted(database):
        print(f"{variable} = {database[variable]}")

    print("\nRECOVERY COMPLETED SUCCESSFULLY.")
    print("=" * 55)

    return database


if __name__ == "__main__":

    log_file = Path(__file__).parent / "log_input.txt"

    recover_database(log_file)
'''

with open("recovery_manager.py", "w") as file:
    file.write(python_code)


# ============================================================
# FILE 3: EXECUTE THE PYTHON SCRIPT
# ============================================================

print("Executing recovery_manager.py...\n")

result = subprocess.run(
    [sys.executable, "recovery_manager.py"],
    capture_output=True,
    text=True
)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Recovery manager execution failed.")

print(result.stdout)


# ============================================================
# FILE 4: recovery_output.png
# Save a screenshot-style image of console output
# ============================================================

try:
    import matplotlib.pyplot as plt
except ImportError:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "matplotlib"]
    )
    import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 10))

ax.set_facecolor("#111111")
fig.patch.set_facecolor("#111111")

ax.axis("off")

ax.text(
    0.03,
    0.97,
    result.stdout,
    transform=ax.transAxes,
    fontsize=11,
    fontfamily="monospace",
    color="#00FF66",
    verticalalignment="top",
    linespacing=1.5
)

plt.savefig(
    "recovery_output.png",
    dpi=180,
    bbox_inches="tight",
    facecolor="#111111"
)

plt.show()
plt.close()


# ============================================================
# FILE 5: recovery_logic.pdf
# Brief report (under 200 words)
# ============================================================

try:
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import getSampleStyleSheet
    from reportlab.platypus import (
        SimpleDocTemplate,
        Paragraph,
        Spacer
    )
except ImportError:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "reportlab"]
    )

    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import getSampleStyleSheet
    from reportlab.platypus import (
        SimpleDocTemplate,
        Paragraph,
        Spacer
    )

report = SimpleDocTemplate(
    "recovery_logic.pdf",
    pagesize=A4
)

styles = getSampleStyleSheet()

story = []

story.append(
    Paragraph(
        "TASK 1: LOG-BASED UNDO AND REDO OPERATIONS",
        styles["Title"]
    )
)

story.append(Spacer(1, 20))

report_text = """
<strong>Database Recovery Logic</strong><br/><br/>

A database recovery manager uses transaction logs to restore
the database to a consistent state after a system failure.

Transactions containing a START record and a COMMIT record
are considered completed transactions. Their operations
must be REDONE by applying the new values (v2), ensuring
that committed changes are preserved.

A transaction containing a START record but no COMMIT
or ABORT record is considered incomplete. It must be
UNDONE because the transaction did not successfully
complete before the system failure. Its changes cannot
be treated as permanent. Therefore, the recovery manager
restores the original values (v1) to maintain database
consistency.

During recovery, committed transactions are processed
in the REDO phase, while incomplete transactions are
processed in reverse log order during the UNDO phase.

This process ensures that committed changes are retained
and incomplete transactions do not leave partial updates
in the database.
"""

story.append(
    Paragraph(report_text, styles["BodyText"])
)

report.build(story)


# ============================================================
# FINAL: DISPLAY GENERATED FILES
# ============================================================

print("\n" + "=" * 55)
print("ALL ASSIGNMENT FILES CREATED SUCCESSFULLY")
print("=" * 55)

files = [
    "recovery_manager.py",
    "log_input.txt",
    "recovery_output.png",
    "recovery_logic.pdf"
]

for filename in files:
    print(f"[SAVED] /content/{filename}")

print("\nCheck the Files panel on the left side of Google Colab.")